# Homework: Airbnb Database Analysis

## Group number:

Register your group [here](https://docs.google.com/spreadsheets/d/1x3htD8e3jwgOb4CQckgrhO8l51WJmPDXVjZWM1cb8_o/edit?usp=sharing)

## Group members:
1. `Part A: 20250376 | Daniel Ribeiro`
2. `Part B: 20250407 | Marta Feital`
3. `Part C: 20250364 | Beatriz Pinto`
4. `Part D: 20250434 | Beatriz Correia`
5. `Part E: 20250389 | Pedro Rei`

# 1. Setup & Connection

In [105]:
from datetime import datetime
from bson.decimal128 import Decimal128
from pprint import pprint
import time
import warnings
from bson.objectid import ObjectId
from unidecode import unidecode

from pymongo import MongoClient, UpdateOne

In [106]:
warnings.filterwarnings("ignore")

user="AzureDiamond"
password="hunter2"
host="localhost"
port="27017"
protocol="mongodb"

client = MongoClient(f"{protocol}://{user}:{password}@{host}:{port}")

# Database check
db = client.sample_airbnb 
collection = db["listingsAndReviews_HW2_new"]
hosts = db["hosts"]
reviews = db["reviews"]

print(f"Database info: {db}\n")
db.name 



Database info: Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_airbnb')



'sample_airbnb'

[PyMongo documentation](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html)

# 2. Data Quality Analysis

## 2.1 Database Overview

In [107]:
# Collections in the database
collection_list = db.list_collection_names()

# Select main collection explicitly
collection_name = "listingsAndReviews_HW2_new"

# Get sample document and fields
sample_doc = collection.find_one()
fields = list(sample_doc.keys()) if sample_doc else []

# Print info
print(f"The database contains {len(collection_list)} collections")
print(f"All collections: {collection_list}")
print(f"Collection '{collection_name}' contains {collection.count_documents({})} documents")
print(f"Fields in collection '{collection_name}': {fields}")
print("\n--- Collections Overview ---")
for col in collection_list:
    print(f"{col}: {db[col].count_documents({})} documents")

The database contains 1 collections
All collections: ['listingsAndReviews_HW2_new']
Collection 'listingsAndReviews_HW2_new' contains 5555 documents
Fields in collection 'listingsAndReviews_HW2_new': ['_id', 'listing_url', 'name', 'summary', 'space', 'description', 'neighborhood_overview', 'notes', 'transit', 'access', 'interaction', 'house_rules', 'property_type', 'room_type', 'bed_type', 'minimum_nights', 'maximum_nights', 'cancellation_policy', 'last_scraped', 'calendar_last_scraped', 'first_review', 'last_review', 'accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'bathrooms', 'amenities', 'price', 'security_deposit', 'cleaning_fee', 'extra_people', 'guests_included', 'images', 'host', 'address', 'availability', 'review_scores', 'reviews', 'transactions', 'review_scores_rating']

--- Collections Overview ---
listingsAndReviews_HW2_new: 5555 documents


#### Collections

The database contains **1 collections**:

Each document in the collection represents an **Airbnb listing** and stores several types of information in a single document. The working collection for this project is **`listingsAndReviews_HW2_new`**.

#### Main Fields

| Category | Field Name | Description |
| :--- | :--- | :--- |
| **Basic Information** | `_id`, `listing_url`, `name`, `summary`, `space`, `description`, `neighborhood_overview` | Identification, URL, and general textual descriptions of the listing. |
| **Property Characteristics** | `property_type`, `room_type`, `bed_type`, `accommodates`, `bedrooms`, `beds`, `bathrooms` | Physical attributes, capacity, and layout of the property. |
| **Booking Rules** | `minimum_nights`, `maximum_nights`, `cancellation_policy`, `house_rules` | Requirements, limits, and policies for stay and behavior. |
| **Pricing Information** | `price`, `security_deposit`, `cleaning_fee`, `extra_people`, `guests_included` | Detailed cost breakdown, fees, and guest capacity limits. |
| **Reviews and Ratings** | `number_of_reviews`, `first_review`, `last_review`, `review_scores`, `review_scores_rating`, `reviews` | Feedback data, history of reviews, and overall quality metrics. |
| **Host and Location** | `host`, `address` | Information about the listing owner and the geographic location. |
| **Other Fields** | `availability`, `images`, `transactions`, `amenities` | Dynamic status, visual assets, and provided facilities. |

#### Modeling Observation

The collection uses a **highly embedded structure**, meaning that several related pieces of information are stored directly inside each listing document.

Examples:
- `host` is stored as an embedded document
- `address` is stored as an embedded document
- `reviews` is stored as an array of embedded subdocuments

This structure is convenient for retrieving all listing information at once, but it may also create some issues:
- very large documents, especially when listings have many reviews
- repeated host information across multiple listings
- potentially less efficient updates and queries for large embedded arrays

#### Project Implication

Because of this structure, the database is a good candidate for **redesign**, especially in:
- separating large and growing fields such as `reviews`
- reducing redundancy for repeated entities such as `host`
- improving performance with more appropriate indexing

## 2.2 Type & Missing Values Analysis

In [108]:
def check_all_field_types_with_nulls(db, collection_name, fields):
    collection = db[collection_name]
    total_docs = collection.count_documents({})
    
    results = {}
    
    for field in fields:
        # Count data types present in the field
        pipeline = [
            {
                "$group": {
                    "_id": {"$type": f"${field}"},
                    "count": {"$sum": 1}
                }
            }
        ]
        types = list(collection.aggregate(pipeline))
        
        # Count null or missing values
        null_or_missing_count = collection.count_documents({
            "$or": [
                {field: None},
                {field: {"$exists": False}}
            ]
        })
        
        results[field] = {
            "types": types,
            "null_or_missing_count": null_or_missing_count,
            "null_or_missing_percentage": (null_or_missing_count / total_docs) * 100
        }
    
    return results

In [109]:
types_info = check_all_field_types_with_nulls(db, collection_name, fields)

for field, info in types_info.items():
    print(f"\nField: {field}")
    
    for t in info["types"]:
        print(f"  Type: {t['_id']} | Count: {t['count']}")
    
    print(f"  Null or missing count: {info['null_or_missing_count']}")
    print(f"  Null or missing percentage: {info['null_or_missing_percentage']:.2f}%")


Field: _id
  Type: string | Count: 5555
  Null or missing count: 0
  Null or missing percentage: 0.00%

Field: listing_url
  Type: string | Count: 5555
  Null or missing count: 0
  Null or missing percentage: 0.00%

Field: name
  Type: string | Count: 5547
  Type: missing | Count: 8
  Null or missing count: 8
  Null or missing percentage: 0.14%

Field: summary
  Type: string | Count: 5297
  Type: missing | Count: 258
  Null or missing count: 258
  Null or missing percentage: 4.64%

Field: space
  Type: string | Count: 3929
  Type: missing | Count: 1626
  Null or missing count: 1626
  Null or missing percentage: 29.27%

Field: description
  Type: string | Count: 5460
  Type: missing | Count: 95
  Null or missing count: 95
  Null or missing percentage: 1.71%

Field: neighborhood_overview
  Type: string | Count: 3314
  Type: missing | Count: 2241
  Null or missing count: 2241
  Null or missing percentage: 40.34%

Field: notes
  Type: string | Count: 2475
  Type: missing | Count: 3080
  N

## 2.3 Data Type Conversion and Missing Value Imputation

### 2.3.1 Data Type Conversion

In [110]:
# Converter extra_people: decimal → int 
result = collection.update_many(
    {"extra_people": {"$type": "decimal"}},   
    [{"$set": {"extra_people": {"$toInt": "$extra_people"}}}]
)
print(f"extra_people convertidos: {result.modified_count}")

print("\nAfter — extra_people type:")
for r in collection.aggregate([
    {"$group": {"_id": {"$type": "$extra_people"}, "count": {"$sum": 1}}}
]):
    print(r)

extra_people convertidos: 5555

After — extra_people type:
{'_id': 'int', 'count': 5555}


### 2.3.2 Null and Missing Value Imputation

In [111]:
null_amenities = collection.count_documents({"amenities": None})
missing_security_deposit = collection.count_documents({"security_deposit": None})
missing_cleaning_fee = collection.count_documents({"cleaning_fee": None})

print("Null amenities:", null_amenities)
print("Missing security_deposit:", missing_security_deposit)
print("Missing cleaning_fee:", missing_cleaning_fee)

Null amenities: 30
Missing security_deposit: 2084
Missing cleaning_fee: 1531


In [112]:
# Fix null values in amenities
result = collection.update_many(
    {"amenities": None},
    {"$set": {"amenities": []}}
)

print("Documents modified:", result.modified_count)

Documents modified: 30


In [113]:
# Fix missing security_deposit
result = collection.update_many(
    {"security_deposit": None},
    {"$set": {"security_deposit": Decimal128("0.00")}}
)

print("Documents modified:", result.modified_count)

Documents modified: 2084


In [114]:
result = collection.update_many(
    {"cleaning_fee": None},
    {"$set": {"cleaning_fee": Decimal128("0.00")}}
)

print("Documents modified:", result.modified_count)

Documents modified: 1531


## 2.4 Duplicate Analysis

In [115]:
def find_duplicates(db, collection_name, fields):
    collection = db[collection_name]

    # Allow both a single field and a list of fields
    if isinstance(fields, str):
        fields = [fields]

    # Only consider documents where all fields exist and are not null
    match_condition = {
        field: {"$exists": True, "$ne": None}
        for field in fields
    }

    # Build composite key for grouping
    group_id = {
        field.replace(".", "_"): f"${field}"
        for field in fields
    }

    pipeline = [
        {"$match": match_condition},
        {
            "$group": {
                "_id": group_id,
                "count": {"$sum": 1},
                "docs": {"$push": "$_id"}
            }
        },
        {"$match": {"count": {"$gt": 1}}},
        {"$sort": {"count": -1}}
    ]

    return list(collection.aggregate(pipeline))


def print_duplicate_summary(results, label, max_examples=5):
    print(f"\n{'='*60}")
    print(f"Duplicate analysis: {label}")
    print(f"{'='*60}")

    if not results:
        print("No duplicates found.")
        return

    print(f"Number of duplicate groups: {len(results)}")
    print(f"Total documents involved: {sum(r['count'] for r in results)}")

    print("\nTop examples:")
    for r in results[:max_examples]:
        print(f"Value / Key: {r['_id']}")
        print(f"Count: {r['count']}")
        print(f"Document IDs: {r['docs'][:5]}")
        print("-" * 40)

In [116]:
duplicates_listing_url = find_duplicates(db, collection_name, "listing_url")
print_duplicate_summary(duplicates_listing_url, "listing_url")

duplicates_name_host = find_duplicates(
    db,
    collection_name,
    ["name", "host.host_id"]
)
print_duplicate_summary(duplicates_name_host, "name + host.host_id")

duplicates_name_street = find_duplicates(
    db,
    collection_name,
    ["name", "address.street"]
)
print_duplicate_summary(duplicates_name_street, "name + address.street")

duplicates_name_host_street = find_duplicates(
    db,
    collection_name,
    ["name", "host.host_id", "address.street"]
)
print_duplicate_summary(duplicates_name_host_street, "name + host.host_id + address.street")


Duplicate analysis: listing_url
No duplicates found.

Duplicate analysis: name + host.host_id
Number of duplicate groups: 2
Total documents involved: 4

Top examples:
Value / Key: {'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'host_host_id': '74079684'}
Count: 2
Document IDs: ['32338760', '32338783']
----------------------------------------
Value / Key: {'name': 'Quarto moradia luxo', 'host_host_id': '158889535'}
Count: 2
Document IDs: ['26556751', '26563602']
----------------------------------------

Duplicate analysis: name + address.street
Number of duplicate groups: 5
Total documents involved: 10

Top examples:
Value / Key: {'name': 'İstanbul Birden fazla bölümden oluşan bina', 'address_street': 'Beşiktaş, İstanbul, Turkey'}
Count: 2
Document IDs: ['28221594', '29224880']
----------------------------------------
Value / Key: {'name': 'Feel like home', 'address_street': 'Montréal, Québec, Canada'}
Count: 2
Document IDs: ['11566403', '4840858']
-

In [117]:
if duplicates_name_host_street:
    example = duplicates_name_host_street[0]
    doc_ids = example["docs"]

    docs = list(db[collection_name].find({"_id": {"$in": doc_ids}}))

    print("\nInspecting one duplicate group:")
    print("Duplicate key:", example["_id"])
    print("Count:", example["count"])

    for d in docs:
        print("\nDocument _id:", d.get("_id"))
        print("listing_url:", d.get("listing_url"))
        print("name:", d.get("name"))
        print("host_id:", d.get("host", {}).get("host_id"))
        print("street:", d.get("address", {}).get("street"))


Inspecting one duplicate group:
Duplicate key: {'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'host_host_id': '74079684', 'address_street': 'Honolulu, HI, United States'}
Count: 2

Document _id: 32338760
listing_url: https://www.airbnb.com/rooms/32338760
name: Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!
host_id: 74079684
street: Honolulu, HI, United States

Document _id: 32338783
listing_url: https://www.airbnb.com/rooms/32338783
name: Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!
host_id: 74079684
street: Honolulu, HI, United States


| Check | Fields Used                            | Number of Duplicate Groups | Number of Documents Involved | Observations                                                                                                             |
| ----- | -------------------------------------- | -------------------------- | ---------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| 1     | `listing_url`                          | 0                          | 0                            | No duplicates found. Each listing URL is unique.                                                                         |
| 2     | `name + host.host_id`                  | 2                          | 4                            | Same host has listings with identical titles, indicating potential repetition.                                           |
| 3     | `name + address.street`                | 5                          | 10                           | Listings share the same title and street, suggesting similar listings at the same location.                              |
| 4     | `name + host.host_id + address.street` | 2                          | 4                            | Listings share the same title, host, and street. Strong indication of highly similar listings, but not exact duplicates. |


Although the table summarizes the number of potential duplicates, a closer inspection reveals important context that cannot be inferred from the numbers alone. The cases identified with composite keys represent listings that are highly similar but not exact duplicates. For example, the same host may list multiple rooms or units within the same property, sharing the same title and street address while maintaining distinct listing URLs. This indicates intentional, valid entries rather than data errors.

## 2.5 Invalid Values and Outlier Analysis

In [118]:
# =========================================================
# 1. Logical consistency check not covered before
# =========================================================

invalid_nights = collection.count_documents({
    "$expr": {"$gt": ["$minimum_nights", "$maximum_nights"]}
})

print("Invalid night ranges:", invalid_nights)

Invalid night ranges: 0


In [119]:
# =========================================================
# 2. Price distribution statistics
# =========================================================

price_stats = list(collection.aggregate([
    {
        "$group": {
            "_id": None,
            "min": {"$min": "$price"},
            "max": {"$max": "$price"},
            "avg": {"$avg": "$price"},
            "stdDev": {"$stdDevPop": "$price"}
        }
    }
]))

print(price_stats)

[{'_id': None, 'min': Decimal128('9.00'), 'max': Decimal128('48842.00'), 'avg': Decimal128('276.4275755228505034856700232378002'), 'stdDev': 916.0413171868535}]


In [120]:
# =========================================================
# 3. Price outliers using IQR
# =========================================================

price_percentiles = list(collection.aggregate([
    {
        "$group": {
            "_id": None,
            "q1": {
                "$percentile": {
                    "input": "$price",
                    "p": [0.25],
                    "method": "approximate"
                }
            },
            "q3": {
                "$percentile": {
                    "input": "$price",
                    "p": [0.75],
                    "method": "approximate"
                }
            }
        }
    }
]))

q1 = price_percentiles[0]["q1"][0]
q3 = price_percentiles[0]["q3"][0]

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

price_outliers_count = collection.count_documents({
    "price": {"$gt": upper_bound}
})

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of price outliers:", price_outliers_count)

Q1: 70.0
Q3: 282.0
IQR: 212.0
Lower bound: -248.0
Upper bound: 600.0
Number of price outliers: 366


In [121]:
# =========================================================
# 4. Manual inspection of highest prices
# =========================================================

top_prices = collection.find(
    {},
    {"name": 1, "price": 1, "address.market": 1}
).sort("price", -1).limit(10)

print("\nTop 10 highest prices:")
for doc in top_prices:
    print(doc)


Top 10 highest prices:
{'_id': '20275354', 'name': 'İstanbul un kalbi sisli. Center of istanbul sisli', 'price': Decimal128('48842.00'), 'address': {'market': 'Istanbul'}}
{'_id': '13997910', 'name': 'Apartamento de luxo em Copacabana - 4 quartos', 'price': Decimal128('11190.00'), 'address': {'market': 'Rio De Janeiro'}}
{'_id': '30327756', 'name': '5 PEOPLE ROOM ( 1 TRIP and 1 DOUBLE)', 'price': Decimal128('10001.00'), 'address': {'market': 'Hong Kong'}}
{'_id': '6147746', 'name': 'Stunning Waterfront Marina bay house in Sai Kung', 'price': Decimal128('7002.00'), 'address': {'market': 'Hong Kong'}}
{'_id': '20517090', 'name': 'Deslumbrante apartamento na AV.Atlantica', 'price': Decimal128('6002.00'), 'address': {'market': 'Rio De Janeiro'}}
{'_id': '2271702', 'name': 'LUXURY HOUSE IN BARRA DA TIJUCA', 'price': Decimal128('5502.00'), 'address': {'market': 'Rio De Janeiro'}}
{'_id': '24847281', 'name': '鮀城小家-感受不一样的异地之旅', 'price': Decimal128('4828.00'), 'address': {'market': 'Hong Kong'

In [122]:
# =========================================================
# 5. Manual inspection of highest accommodates
# =========================================================

top_accommodates = collection.find(
    {},
    {"name": 1, "accommodates": 1, "address.market": 1}
).sort("accommodates", -1).limit(10)

print("\nTop 10 highest accommodates:")
for doc in top_accommodates:
    print(doc)


Top 10 highest accommodates:
{'_id': '19587001', 'name': 'Kaena O Kekai', 'accommodates': 16, 'address': {'market': 'Oahu'}}
{'_id': '2271702', 'name': 'LUXURY HOUSE IN BARRA DA TIJUCA', 'accommodates': 16, 'address': {'market': 'Rio De Janeiro'}}
{'_id': '23644123', 'name': 'Party/Event – Day/Night – Botafogo/Laranjeiras!', 'accommodates': 16, 'address': {'market': 'Rio De Janeiro'}}
{'_id': '20958766', 'name': 'Great Complex of the Cellars', 'accommodates': 16, 'address': {'market': 'Porto'}}
{'_id': '20455499', 'name': 'DOWNTOWN VIP MONTREAL ,HIGH END DECOR,GOOD VALUE..', 'accommodates': 16, 'address': {'market': 'Montreal'}}
{'_id': '23219274', 'name': '❤️ 6-BR APARTMENT w Kitchen + 2 Rooms!', 'accommodates': 16, 'address': {'market': 'Istanbul'}}
{'_id': '27383511', 'name': 'LEGNOCASA', 'accommodates': 16, 'address': {'market': 'Istanbul'}}
{'_id': '23921295', 'name': '-MAISON ALEXANDRA- 3 STOREY WATERFRONT PH FLOOR', 'accommodates': 16, 'address': {'market': 'Montreal'}}
{'_id':

The dataset demonstrates strong logical consistency, as no listings have minimum nights greater than maximum nights. Price values, however, exhibit a highly right-skewed distribution, with a small proportion of extremely high listings, particularly in premium or luxury markets such as Istanbul, Hong Kong, and Rio de Janeiro. Using the IQR method, approximately 545 listings were identified as price outliers, while accommodation capacities remain within plausible ranges, with a maximum of 16 guests across several locations. These findings indicate that extreme values are generally valid and should be treated as outliers rather than errors.

### 2.5.1 Consistency Checks

To assess logical coherence between related attributes, several consistency checks were performed across the dataset.

In [123]:
inconsistent_accommodates_beds = collection.count_documents({
    "$expr": {"$lt": ["$accommodates", "$beds"]}
})

potential_inconsistent_beds_bedrooms = collection.count_documents({
    "$expr": {"$lt": ["$beds", "$bedrooms"]}
})

inconsistent_reviews_1 = collection.count_documents({
    "number_of_reviews": 0,
    "first_review": {"$ne": None}
})

inconsistent_reviews_2 = collection.count_documents({
    "$expr": {"$lt": ["$last_review", "$first_review"]}
})

In [124]:
print("\n=== Consistency Checks ===")

print("\nCapacity consistency:")
print("Accommodates < beds:", inconsistent_accommodates_beds)

print("\nBedroom vs Beds (potential inconsistency):")
print("Beds < bedrooms:", potential_inconsistent_beds_bedrooms)

print("\nReview consistency:")
print("0 reviews but has first_review:", inconsistent_reviews_1)
print("last_review < first_review:", inconsistent_reviews_2)


=== Consistency Checks ===

Capacity consistency:
Accommodates < beds: 47

Bedroom vs Beds (potential inconsistency):
Beds < bedrooms: 95

Review consistency:
0 reviews but has first_review: 0
last_review < first_review: 0


The consistency analysis shows that:

- most relationships between key attributes are coherent
- review-related metadata is fully consistent
- a limited number of potentially inconsistent cases exist for:
  - `accommodates < beds`
  - `beds < bedrooms`

These cases should be flagged for awareness, but they do not necessarily justify automatic removal without further contextual inspection.

### 2.5.2 Empty Arrays, Null Values, and Cardinality Analysis

In [125]:
empty_amenities = collection.count_documents({
    "amenities": {"$size": 0}
})

missing_amenities = collection.count_documents({
    "amenities": {"$exists": False}
})

null_amenities = collection.count_documents({
    "amenities": None
})

non_empty_amenities = collection.count_documents({
    "amenities": {"$exists": True, "$nin": [[], None]}
})

print("Empty amenities:", empty_amenities)
print("Missing amenities:", missing_amenities)
print("Null amenities:", null_amenities)
print("Non-empty amenities:", non_empty_amenities)

Empty amenities: 30
Missing amenities: 0
Null amenities: 0
Non-empty amenities: 5525


In [126]:
def cardinality_analysis(db, collection_name, field, top_n=10):
    collection = db[collection_name]

    pipeline = [
        {"$group": {"_id": f"${field}", "count": {"$sum": 1}}},
        {"$sort": {"count": -1}}
    ]

    results = list(collection.aggregate(pipeline))

    print(f"\nField: {field}")
    print(f"Unique values: {len(results)}")

    for r in results[:top_n]:
        print(r)


cardinality_analysis(db, collection_name, "property_type")
cardinality_analysis(db, collection_name, "room_type")
cardinality_analysis(db, collection_name, "bed_type")
cardinality_analysis(db, collection_name, "address.market")


Field: property_type
Unique values: 36
{'_id': 'Apartment', 'count': 3626}
{'_id': 'House', 'count': 606}
{'_id': 'Condominium', 'count': 399}
{'_id': 'Serviced apartment', 'count': 185}
{'_id': 'Loft', 'count': 142}
{'_id': 'Townhouse', 'count': 108}
{'_id': 'Guest suite', 'count': 81}
{'_id': 'Bed and breakfast', 'count': 69}
{'_id': 'Boutique hotel', 'count': 53}
{'_id': 'Guesthouse', 'count': 50}

Field: room_type
Unique values: 3
{'_id': 'Entire home/apt', 'count': 3489}
{'_id': 'Private room', 'count': 1983}
{'_id': 'Shared room', 'count': 83}

Field: bed_type
Unique values: 5
{'_id': 'Real Bed', 'count': 5506}
{'_id': 'Pull-out Sofa', 'count': 26}
{'_id': 'Futon', 'count': 10}
{'_id': 'Couch', 'count': 7}
{'_id': 'Airbed', 'count': 6}

Field: address.market
Unique values: 15
{'_id': 'Istanbul', 'count': 660}
{'_id': 'Montreal', 'count': 648}
{'_id': 'Barcelona', 'count': 632}
{'_id': 'Hong Kong', 'count': 619}
{'_id': 'Sydney', 'count': 609}
{'_id': 'New York', 'count': 607}
{'

## 2.6 Task D: Specific Data Quality Checks

This section verifies specific data quality requirements mentioned in the instructions, such as incorrect timestamps, wrong data formats, and incorrect value ranges.

In [127]:
# Reference dates
AIRBNB_FOUNDING = datetime(2008, 8, 1)
SCRAPE_DATE_MAX = datetime(2020, 1, 1)

### 2.6.1 Incorrect Timestamps

In [128]:
print("=" * 60)
print("TIMESTAMP AUDIT - LISTINGS")
print("=" * 60)

date_fields = ["last_scraped", "calendar_last_scraped", "first_review", "last_review"]

for field in date_fields:
    future = collection.count_documents({field: {"$exists": True, "$ne": None, "$gt": SCRAPE_DATE_MAX}})
    too_old = collection.count_documents({field: {"$exists": True, "$ne": None, "$lt": AIRBNB_FOUNDING}})
    print(f"\nField: {field}")
    print(f"  Future dates (>{SCRAPE_DATE_MAX.date()}): {future}")
    print(f"  Pre-Airbnb dates (<{AIRBNB_FOUNDING.date()}): {too_old}")

pipeline = [
    {"$match": {"last_scraped": {"$exists": True}, "calendar_last_scraped": {"$exists": True}}},
    {"$project": {
        "diff_days": {"$divide": [{"$abs": {"$subtract": ["$last_scraped", "$calendar_last_scraped"]}}, 1000 * 60 * 60 * 24]}
    }},
    {"$group": {
        "_id": None,
        "max_diff": {"$max": "$diff_days"},
        "avg_diff": {"$avg": "$diff_days"},
        "over_30_days": {"$sum": {"$cond": [{"$gt": ["$diff_days", 30]}, 1, 0]}}
    }}
]
diff_result = list(collection.aggregate(pipeline))
if diff_result:
    r = diff_result[0]
    print(f"\nlast_scraped vs calendar_last_scraped:")
    print(f"  Max diff: {r['max_diff']:.1f} days")
    print(f"  Avg diff: {r['avg_diff']:.2f} days")
    print(f"  Pairs with > 30 days diff: {r['over_30_days']}")

TIMESTAMP AUDIT - LISTINGS

Field: last_scraped
  Future dates (>2020-01-01): 0
  Pre-Airbnb dates (<2008-08-01): 0

Field: calendar_last_scraped
  Future dates (>2020-01-01): 0
  Pre-Airbnb dates (<2008-08-01): 0

Field: first_review
  Future dates (>2020-01-01): 0
  Pre-Airbnb dates (<2008-08-01): 0

Field: last_review
  Future dates (>2020-01-01): 0
  Pre-Airbnb dates (<2008-08-01): 0

last_scraped vs calendar_last_scraped:
  Max diff: 0.0 days
  Avg diff: 0.00 days
  Pairs with > 30 days diff: 0


In [129]:
print("=" * 60)
print("TIMESTAMP AUDIT - REVIEWS")
print("=" * 60)

future_reviews = reviews.count_documents({"date": {"$gt": SCRAPE_DATE_MAX}})
old_reviews = reviews.count_documents({"date": {"$lt": AIRBNB_FOUNDING}})
no_date = reviews.count_documents({"$or": [{"date": None}, {"date": {"$exists": False}}]})

print(f"Future dates (>{SCRAPE_DATE_MAX.date()}): {future_reviews}")
print(f"Pre-Airbnb dates (<{AIRBNB_FOUNDING.date()}): {old_reviews}")
print(f"Missing date: {no_date}")

TIMESTAMP AUDIT - REVIEWS
Future dates (>2020-01-01): 0
Pre-Airbnb dates (<2008-08-01): 0
Missing date: 0


In [130]:
print("=" * 60)
print("TIMESTAMP AUDIT - REVIEWS")
print("=" * 60)

# 1. Datas no futuro
future_reviews = collection.count_documents({
    "reviews": {
        "$elemMatch": {"date": {"$gt": SCRAPE_DATE_MAX}}
    }
})

# 2. Datas antes do Airbnb
old_reviews = collection.count_documents({
    "reviews": {
        "$elemMatch": {"date": {"$lt": AIRBNB_FOUNDING}}
    }
})

# 3. Reviews sem data
no_date = collection.count_documents({
    "reviews": {
        "$elemMatch": {
            "$or": [
                {"date": None},
                {"date": {"$exists": False}}
            ]
        }
    }
})

print(f"Future dates (>{SCRAPE_DATE_MAX.date()}): {future_reviews}")
print(f"Pre-Airbnb dates (<{AIRBNB_FOUNDING.date()}): {old_reviews}")
print(f"Missing date: {no_date}")

TIMESTAMP AUDIT - REVIEWS
Future dates (>2020-01-01): 0
Pre-Airbnb dates (<2008-08-01): 0
Missing date: 0


### 2.6.3 Incorrect Value Ranges

In [ ]:
print("=" * 60)
print("RANGE AUDIT - review_scores")
print("=" * 60)

out_rating = collection.count_documents({
    "review_scores.review_scores_rating": {"$exists": True, "$ne": None},
    "$or": [{"review_scores.review_scores_rating": {"$lt": 0}}, {"review_scores.review_scores_rating": {"$gt": 100}}]
})
print(f"review_scores_rating outside [0, 100]: {out_rating}")

subscores = ["accuracy", "cleanliness", "checkin", "communication", "location", "value"]
for sub in subscores:
    out = collection.count_documents({
        f"review_scores.review_scores_{sub}": {"$exists": True, "$ne": None},
        "$or": [{f"review_scores.review_scores_{sub}": {"$lt": 0}}, {f"review_scores.review_scores_{sub}": {"$gt": 10}}]
    })
    if out > 0:
        print(f"Review_scores_{sub} outside [0, 10]: {out}")

RANGE AUDIT - review_scores
review_scores_rating outside [0, 100]: 0


In [147]:
print("=" * 60)
print("RANGE AUDIT - Capacity and Nights")
print("=" * 60)

checks = [
    ("accommodates <= 0", {"accommodates": {"$lte": 0}}),
    ("bedrooms < 0", {"bedrooms": {"$lt": 0}}),
    ("beds < 0", {"beds": {"$lt": 0}}),
    ("bathrooms < 0", {"bathrooms": {"$lt": 0}}),
    ("minimum_nights <= 0", {"minimum_nights": {"$lte": 0}}),
    ("maximum_nights <= 0", {"maximum_nights": {"$lte": 0}}),
    ("number_of_reviews < 0", {"number_of_reviews": {"$lt": 0}}),
    ("guests_included <= 0", {"guests_included": {"$lte": 0}}),
    ("extra_people < 0", {"extra_people": {"$lt": 0}}),
    ("price <= 0", {"price": {"$lte": 0}}),
    ("cleaning_fee < 0", {"cleaning_fee": {"$lt": 0}}),
    ("security_deposit < 0", {"security_deposit": {"$lt": 0}}),
]

for label, filt in checks:
    count = collection.count_documents(filt)
    print(f"{label}: {count}")

RANGE AUDIT - Capacity and Nights
accommodates <= 0: 0
bedrooms < 0: 0
beds < 0: 0
bathrooms < 0: 0
minimum_nights <= 0: 0
maximum_nights <= 0: 0
number_of_reviews < 0: 0
guests_included <= 0: 0
extra_people < 0: 0
price <= 0: 0
cleaning_fee < 0: 0
security_deposit < 0: 0


In [148]:
print("=" * 60)
print("RANGE AUDIT - Hosts")
print("=" * 60)

host_checks = [
    ("host_response_rate < 0", {"host.host_response_rate": {"$lt": 0}}),
    ("host_response_rate > 100", {"host.host_response_rate": {"$gt": 100}}),
    ("host_listings_count < 0", {"host.host_listings_count": {"$lt": 0}}),
    ("host_total_listings_count < 0", {"host.host_total_listings_count": {"$lt": 0}}),
    ("listings_count != total_count", {"$expr": {"$ne": ["$host.host_listings_count", "$host.host_total_listings_count"]}}),
]

for label, filt in host_checks:
    count = collection.count_documents(filt)
    print(f"{label}: {count}")

RANGE AUDIT - Hosts
host_response_rate < 0: 0
host_response_rate > 100: 0
host_listings_count < 0: 0
host_total_listings_count < 0: 0
listings_count != total_count: 0


In [149]:
print("=" * 60)
print("RANGE AUDIT - Availability")
print("=" * 60)

avail_checks = [
    ("availability_30 < 0 or > 30", {"$or": [{"availability.availability_30": {"$lt": 0}}, {"availability.availability_30": {"$gt": 30}}]}),
    ("availability_60 < 0 or > 60", {"$or": [{"availability.availability_60": {"$lt": 0}}, {"availability.availability_60": {"$gt": 60}}]}),
    ("availability_90 < 0 or > 90", {"$or": [{"availability.availability_90": {"$lt": 0}}, {"availability.availability_90": {"$gt": 90}}]}),
    ("availability_365 < 0 or > 365", {"$or": [{"availability.availability_365": {"$lt": 0}}, {"availability.availability_365": {"$gt": 365}}]}),
    ("availability_30 > availability_60", {"$expr": {"$gt": ["$availability.availability_30", "$availability.availability_60"]}}),
    ("availability_60 > availability_90", {"$expr": {"$gt": ["$availability.availability_60", "$availability.availability_90"]}}),
    ("availability_90 > availability_365", {"$expr": {"$gt": ["$availability.availability_90", "$availability.availability_365"]}}),
]

for label, filt in avail_checks:
    count = collection.count_documents(filt)
    print(f"{label}: {count}")

RANGE AUDIT - Availability
availability_30 < 0 or > 30: 0
availability_60 < 0 or > 60: 0
availability_90 < 0 or > 90: 0
availability_365 < 0 or > 365: 0
availability_30 > availability_60: 0
availability_60 > availability_90: 0
availability_90 > availability_365: 0


# 3. Data Cleaning

## 3.1 Remove Useless Field (`review_scores_rating`)

In [135]:
result = collection.update_many(
    {"review_scores_rating": {"$exists": True}},
    {"$unset": {"review_scores_rating": ""}}
)
print("Documents modified:", result.modified_count)


remaining = collection.count_documents({"review_scores_rating": {"$exists": True}})
print("Documents still containing review_scores_rating:", remaining)

Documents modified: 5555
Documents still containing review_scores_rating: 0


# 4. Database Redesign

The collection embeds both host data and all reviews within each listing, which causes two main issues. First, unbounded review arrays make listings with many reviews very large, slowing read operations, to address this, the Subset Pattern + Extended Reference Pattern is applied: all reviews are moved to a separate reviews collection with a listing_id reference, while only the three most recent reviews are kept in a reviews_subset field for fast access. Second, duplicated host data occurs because hosts managing multiple listings have repeated attributes across documents, making updates complex and error-prone this is resolved using the Extended Reference Pattern, extracting host data into a dedicated hosts collection keyed by host_id, while each listing retains only the host_id.


## 4.1 Reviews Redesign

In [136]:
doc = collection.find_one(
    {"reviews.0": {"$exists": True}},
    {"_id": 1, "reviews": 1}
)

print("Listing ID:", doc["_id"])
print("Type of reviews:", type(doc["reviews"]))
print("Number of reviews:", len(doc["reviews"]))
print("\nSample review:")
print(doc["reviews"][0])

Listing ID: 10006546
Type of reviews: <class 'list'>
Number of reviews: 51

Sample review:
{'_id': '58663741', 'date': datetime.datetime(2016, 1, 3, 5, 0), 'listing_id': '10006546', 'reviewer_id': '51483096', 'reviewer_name': 'Cátia', 'comments': 'A casa da Ana e do Gonçalo foram o local escolhido para a passagem de ano com um grupo de amigos. Fomos super bem recebidos com uma grande simpatia e predisposição a ajudar com qualquer coisa que fosse necessário.\r\nA casa era ainda melhor do que parecia nas fotos, totalmente equipada, com mantas, aquecedor e tudo o que pudessemos precisar.\r\nA localização não podia ser melhor! Não há melhor do que acordar de manhã e ao virar da esquina estar a ribeira do Porto.'}


In [ ]:
from pymongo import UpdateOne, InsertOne
from datetime import datetime

existing_reviews = db.reviews.count_documents({})

if existing_reviews > 0:
    print(f"A coleção 'reviews' já se encontra populada ({existing_reviews} documentos) — a abortar migração para evitar duplicação de dados.")
else:
    # Definir o tamanho do lote para otimização de memória RAM e I/O
    BATCH_SIZE = 5000 
    
    reviews_to_insert_batch = []
    listings_to_update_batch = []
    processed_count = 0

    # Otimização da query: procurar apenas documentos que contenham o campo 'reviews'
    cursor = collection.find({"reviews": {"$exists": True}}, {"reviews": 1})

    for listing in cursor:
        listing_id = listing["_id"]
        reviews = listing.get("reviews")

        # Cenário A: O campo existe, mas não é uma lista válida ou encontra-se vazio.
        # Prepara a remoção imediata do campo obsoleto.
        if not isinstance(reviews, list) or len(reviews) == 0:
            listings_to_update_batch.append(
                UpdateOne(
                    {"_id": listing_id},
                    {"$unset": {"reviews": ""}}
                )
            )
        else:
            # Cenário B: O documento tem avaliações válidas.
            # Ordenar por data (mais recentes primeiro), salvaguardando datas nulas
            reviews_sorted = sorted(
                reviews,
                key=lambda r: r.get("date") if r.get("date") else datetime.min,
                reverse=True
            )

            # Extrair subconjunto (top 3)
            reviews_subset = [
                {
                    "date": r.get("date"),
                    "reviewer_name": r.get("reviewer_name"),
                    "comments": r.get("comments")
                }
                for r in reviews_sorted[:3]
            ]

            # Preparar as operações de inserção para a nova coleção
            for r in reviews:
                review_doc = r.copy()
                review_doc["listing_id"] = listing_id
                reviews_to_insert_batch.append(InsertOne(review_doc))

            # Preparar a operação de atualização na coleção original
            listings_to_update_batch.append(
                UpdateOne(
                    {"_id": listing_id},
                    {
                        "$set": {"reviews_subset": reviews_subset},
                        "$unset": {"reviews": ""}
                    }
                )
            )

        processed_count += 1

        # Mecanismo de descarga (Flush): Executar as operações em lote
        if processed_count % BATCH_SIZE == 0:
            if listings_to_update_batch:
                collection.bulk_write(listings_to_update_batch)
                listings_to_update_batch.clear()
            
            if reviews_to_insert_batch:
                db.reviews.bulk_write(reviews_to_insert_batch)
                reviews_to_insert_batch.clear()
            
            print(f"Estado: A processar o documento {processed_count}...")

    # Mecanismo de descarga final (Tail flush): Processar os registos remanescentes do último lote
    if listings_to_update_batch:
        collection.bulk_write(listings_to_update_batch)
    if reviews_to_insert_batch:
        db.reviews.bulk_write(reviews_to_insert_batch)

    print(f"Migração concluída com sucesso. Total de documentos da coleção original avaliados: {processed_count}.")

Migração concluída com sucesso. Total de documentos da coleção original avaliados: 3923.


In [139]:
doc = collection.find_one({"reviews_subset": {"$exists": True}})
print(len(doc["reviews_subset"]))
print("Total reviews:", db.reviews.count_documents({}))

3
Total reviews: 149792


## 4.2 Hosts Redesign

In [140]:
total_with_host = collection.count_documents({"host": {"$exists": True, "$ne": None}})
print("Listings with host:", total_with_host)

unique_hosts = list(collection.aggregate([
    {"$match": {"host.host_id": {"$exists": True, "$ne": None}}},
    {"$group": {"_id": "$host.host_id"}},
    {"$count": "total_unique_hosts"}
]))

print("Unique hosts:", unique_hosts[0]["total_unique_hosts"] if unique_hosts else 0)

Listings with host: 5555
Unique hosts: 5104


In [141]:
doc = collection.find_one(
    {"host": {"$exists": True}},
    {"_id": 1, "host": 1}
)

print("Listing ID:", doc["_id"])
print("Type of host:", type(doc["host"]))
print("\nHost sample:")
print(doc["host"])

Listing ID: 10006546
Type of host: <class 'dict'>

Host sample:
{'host_id': '51399391', 'host_url': 'https://www.airbnb.com/users/show/51399391', 'host_name': 'Ana&Gonçalo', 'host_location': 'Porto, Porto District, Portugal', 'host_about': 'Gostamos de passear, de viajar, de conhecer pessoas e locais novos, gostamos de desporto e animais! Vivemos na cidade mais linda do mundo!!!', 'host_response_time': 'within an hour', 'host_thumbnail_url': 'https://a0.muscache.com/im/pictures/fab79f25-2e10-4f0f-9711-663cb69dc7d8.jpg?aki_policy=profile_small', 'host_picture_url': 'https://a0.muscache.com/im/pictures/fab79f25-2e10-4f0f-9711-663cb69dc7d8.jpg?aki_policy=profile_x_medium', 'host_neighbourhood': '', 'host_response_rate': 100, 'host_is_superhost': False, 'host_has_profile_pic': True, 'host_identity_verified': True, 'host_listings_count': 3, 'host_total_listings_count': 3, 'host_verifications': ['email', 'phone', 'reviews', 'jumio', 'offline_government_id', 'government_id']}


In [142]:
existing_hosts = db.hosts.count_documents({})
if existing_hosts > 0:
    print(f"Hosts collection already populated ({existing_hosts} docs) — skipping migration.")
else:
    hosts_dict = {}

    for listing in collection.find({}, {"host": 1}):
        host = listing.get("host")

        if not isinstance(host, dict):
            continue

        host_id = host.get("host_id")
        if not host_id:
            continue

        # guardar host único
        hosts_dict[host_id] = host

    # preparar lista final de hosts
    hosts_to_insert = list(hosts_dict.values())

    # inserir nova coleção
    if hosts_to_insert:
        db.hosts.insert_many(hosts_to_insert)
        print("Inserted hosts:", len(hosts_to_insert))
    else:
        print("No hosts found to insert (hosts may have already been migrated from listings).")

    # atualizar listings: guardar host_id e remover host
    result = collection.update_many(
        {"host.host_id": {"$exists": True}},
        [
            {
                "$set": {
                    "host_id": "$host.host_id"
                }
            },
            {
                "$unset": "host"
            }
        ]
    )
    print("Listings updated:", result.modified_count)


Inserted hosts: 5104
Listings updated: 5555


In [143]:
print("Hosts collection count:", db.hosts.count_documents({}))
print("Listings with host_id:", collection.count_documents({"host_id": {"$exists": True}}))
print("Listings still containing host:", collection.count_documents({"host": {"$exists": True}}))

Hosts collection count: 5104
Listings with host_id: 5555
Listings still containing host: 0


## 4.3 Post-Redesign Duplicate Verification

After the database redesign, the `host` embedded document was extracted into a separate `hosts` collection.  
Listings now reference hosts via the field `host_id` (flat string), **not** `host.host_id` (embedded path).

The duplicate analysis in **Section 2.4** used `host.host_id`, which was correct at that time.  
Here we re-run the same check post-redesign using `host_id` to confirm the findings remain valid.

**Result:** The same 2 duplicate groups are confirmed. Each pair has a different `listing_url`,  
meaning they are **distinct Airbnb listings** (e.g. different rooms in the same property), not data errors.  
**Decision: No documents were removed.**

In [144]:
host_embedded = collection.count_documents({"host": {"$exists": True}})
host_ref = collection.count_documents({"host_id": {"$exists": True}})
print("=== Field structure after redesign ===")
print(f"Documents with 'host' (embedded) : {host_embedded}")
print(f"Documents with 'host_id' (ref)   : {host_ref}")
print()

duplicates_post = find_duplicates(
    db,
    collection_name,
    ["name", "host_id", "address.street"]
)
print_duplicate_summary(duplicates_post, "name + host_id + address.street (post-redesign)")

if duplicates_post:
    print("\n=== Inspection ===")
    for group in duplicates_post:
        docs = list(db[collection_name].find(
            {"_id": {"$in": group["docs"]}},
            {"_id": 1, "name": 1, "listing_url": 1, "host_id": 1, "address.street": 1}
        ))
        print(f"\nGroup key: {group['_id']}")
        for d in docs:
            print(f"  _id: {d['_id']} | url: {d.get('listing_url', 'N/A')}")
    print("\n-> Different listing_url values confirm these are distinct listings, not true duplicates.")
else:
    print("No duplicate groups found post-redesign.")

=== Field structure after redesign ===
Documents with 'host' (embedded) : 0
Documents with 'host_id' (ref)   : 5555


Duplicate analysis: name + host_id + address.street (post-redesign)
Number of duplicate groups: 2
Total documents involved: 4

Top examples:
Value / Key: {'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'host_id': '74079684', 'address_street': 'Honolulu, HI, United States'}
Count: 2
Document IDs: ['32338760', '32338783']
----------------------------------------
Value / Key: {'name': 'Quarto moradia luxo', 'host_id': '158889535', 'address_street': 'Sanguedo, Aveiro, Portugal'}
Count: 2
Document IDs: ['26556751', '26563602']
----------------------------------------

=== Inspection ===

Group key: {'name': 'Great Location In Wakiki, Walking Distance To Beach, Shopping, Dining!', 'host_id': '74079684', 'address_street': 'Honolulu, HI, United States'}
  _id: 32338760 | url: https://www.airbnb.com/rooms/32338760
  _id: 32338783 | url: https:/

## 4.4 Outlier & Inconsistency Verification

After completing the database redesign (separating `hosts` and `reviews` into dedicated collections), we can now verify the outlier and inconsistency decisions. 

The decisions below are grouped into two categories:

1. **Global decisions** — apply uniformly to the entire database before any task-specific analysis.
2. **Task-specific inclusion criteria** — define which records are used per task, based on the research question.

These two categories are intentionally separate: global decisions ensure data consistency across all tasks,
while task-specific filters reflect different analytical scopes, which is standard practice.

---

## Part 1 — Global Decisions (apply to D1, D2 and D3)

### `price` — 545 outliers detected (IQR upper bound: 597.78)

Outliers were detected using the IQR method. Manual inspection of the top values confirmed
these correspond to genuine premium or luxury listings (Istanbul, Hong Kong, Rio de Janeiro).

**Decision:** Outliers are **retained** across all tasks.
They represent valid market variation. Box plots and aggregations will naturally reflect the full distribution.

---

### `cleaning_fee` — 27.56% missing values

Missing `cleaning_fee` was imputed to `Decimal128("0.00")` in Section 3.1,
under the assumption that a missing value means no cleaning fee is charged.

**Decision:** `cleaning_fee = 0` is used uniformly in all price-per-person calculations (D1, D3).

---

### `extra_people` — stored as `Decimal128`, converted to `int`

This field represents a surcharge per additional guest and was stored as a decimal type.
It was converted to `int` in Section 2.3.1.

**Decision:** The converted `int` value is used consistently across D1 and D3.

---

### `accommodates < beds` — 47 cases

47 listings have more beds than the number of people they accommodate.
This likely reflects flexible sleeping arrangements (e.g. bunk beds, sofa beds).

**Decision:** These listings are **retained**. The inconsistency does not affect price-per-person
calculations, which rely on `guests_included` and the 2-guest assumption stated in D1/D3.

---

## Part 2 — Task-Specific Inclusion Criteria

These are not data quality decisions — they define the relevant population for each research question.
Different tasks may legitimately include different subsets of listings.

---

### D1 — Host Type and Pricing

| Criterion | Value | Reason |
|-----------|-------|--------|
| Cities | Hong Kong, Montreal, Barcelona | Required by the task specification |
| `review_scores_rating` | Not required | D1 analyses pricing only — satisfaction is not measured here |
| `host_response_time/rate` | Not required | Not relevant to D1 |

All valid listings in the three specified cities are included.

---

### D2 — Host Features Impact on Satisfaction

| Criterion | Value | Reason |
|-----------|-------|--------|
| Cities | All markets | No geographic restriction in the task |
| `review_scores_rating` | Must be present | It is the satisfaction metric — cannot be imputed |
| `host_response_time` | Must be present **for that attribute only** | Required to classify fast/slow responders |
| `host_response_rate` | Must be present **for that attribute only** | Required for binary split |
| `host_is_superhost` | All hosts | Boolean field, always populated — no exclusion needed |
| `host_identity_verified` | All hosts | Boolean field, always populated — no exclusion needed |
| `cancellation_policy` | All listings | Always populated — no exclusion needed |

> Note: The 1381 hosts (27.06%) without `host_response_time`/`host_response_rate` are excluded
> **only for the response-related attributes**. They remain valid for the other three attributes
> (superhost status, identity verification, cancellation policy) analysed in D2.

---

### D3 — Property Popularity and Satisfaction

| Criterion | Value | Reason |
|-----------|-------|--------|
| Host type | Professional hosts only (> 1 property) | Required by the task specification |
| `review_scores_rating` | Must be present | Required to classify properties into satisfaction categories |
| `availability.*` | No exclusions needed | Fully populated across all listings |

> The 1474 listings without `review_scores_rating` (26.53%) are excluded from D3.
> These are newly listed or unreviewed properties and cannot be assigned a satisfaction category.

---

## Summary

| Field | Global Decision | D1 | D2 | D3 |
|-------|----------------|----|----|----|
| `price` outliers | Retain (valid luxury listings) | All cities incl. | All incl. | All incl. |
| `cleaning_fee` missing | Imputed to 0 | Applied | — | Applied |
| `extra_people` type | Converted to int | Applied | — | Applied |
| `accommodates < beds` | Retain | Retained | Retained | Retained |
| `review_scores_rating` null | No global fix | Not required | Excluded | Excluded |
| `host_response_time/rate` null | No global fix | Not required | Excluded per attr. | Not required |
| City filter | — | HK, MTL, BCN only | All markets | All markets |
| Host type filter | — | All hosts | All hosts | Professional only |

In [145]:
total_listings = collection.count_documents({})
total_hosts    = hosts.count_documents({})

print("=" * 55)
print("  GLOBAL DECISIONS — verification")
print("=" * 55)

# 1. Price outliers
from bson.decimal128 import Decimal128
price_outliers = collection.count_documents({"price": {"$gt": Decimal128("597.78")}})
print(f"Price outliers (> 597.78) retained : {price_outliers} / {total_listings} ({price_outliers/total_listings*100:.1f}%)")

# 2. Missing cleaning_fee (should be 0 after imputation)
missing_cf = collection.count_documents({
    "$or": [{"cleaning_fee": {"$exists": False}}, {"cleaning_fee": None}]
})
print(f"Missing cleaning_fee post-imputation: {missing_cf}  (expected: 0)")

# 3. accommodates < beds
inconsistent_beds = collection.count_documents({
    "$expr": {"$lt": ["$accommodates", "$beds"]}
})
print(f"accommodates < beds (retained)      : {inconsistent_beds}")

print()
print("=" * 55)
print("  D1 — city populations")
print("=" * 55)

for city in ["Hong Kong", "Montreal", "Barcelona"]:
    n = collection.count_documents({"address.market": city})
    print(f"  {city}: {n} listings")

print()
print("=" * 55)
print("  D2 — inclusion criteria")
print("=" * 55)

# Listings with review_scores_rating present
has_rating = collection.count_documents({
    "review_scores.review_scores_rating": {"$exists": True, "$ne": None}
})
print(f"Listings with review_scores_rating  : {has_rating} / {total_listings} ({has_rating/total_listings*100:.1f}%)")

# Hosts available per attribute
for attr in ["host_response_time", "host_response_rate", "host_is_superhost",
             "host_identity_verified"]:
    available = hosts.count_documents({attr: {"$exists": True, "$ne": None}})
    print(f"Hosts with {attr:<30}: {available} / {total_hosts} ({available/total_hosts*100:.1f}%)")

# Listings with cancellation_policy
has_cancel = collection.count_documents({"cancellation_policy": {"$exists": True, "$ne": None}})
print(f"Listings with cancellation_policy   : {has_cancel} / {total_listings} ({has_cancel/total_listings*100:.1f}%)")

print()
print("=" * 55)
print("  D3 — inclusion criteria")
print("=" * 55)

# Professional hosts (listings_count > 1)
prof_hosts = hosts.count_documents({"host_listings_count": {"$gt": 1}})
print(f"Professional hosts (>1 listing)     : {prof_hosts} / {total_hosts} ({prof_hosts/total_hosts*100:.1f}%)")

# Listings from professional hosts with ratings
pipeline_d3 = [
    {"$lookup": {
        "from": "hosts",
        "localField": "host_id",
        "foreignField": "host_id",
        "as": "host_info"
    }},
    {"$match": {
        "host_info.host_listings_count": {"$gt": 1},
        "review_scores.review_scores_rating": {"$exists": True, "$ne": None}
    }},
    {"$count": "total"}
]
result_d3 = list(collection.aggregate(pipeline_d3))
d3_count = result_d3[0]["total"] if result_d3 else 0
print(f"Listings eligible for D3 analysis   : {d3_count}")

# Availability completeness
for av_field in ["availability.availability_30", "availability.availability_60",
                 "availability.availability_90", "availability.availability_365"]:
    missing_av = collection.count_documents({
        "$or": [{av_field: None}, {av_field: {"$exists": False}}]
    })
    label = av_field.split(".")[1]
    print(f"Missing {label:<25}: {missing_av}  (expected: 0)")

  GLOBAL DECISIONS — verification
Price outliers (> 597.78) retained : 375 / 5555 (6.8%)
Missing cleaning_fee post-imputation: 0  (expected: 0)
accommodates < beds (retained)      : 47

  D1 — city populations
  Hong Kong: 619 listings
  Montreal: 648 listings
  Barcelona: 632 listings

  D2 — inclusion criteria
Listings with review_scores_rating  : 4081 / 5555 (73.5%)
Hosts with host_response_time            : 3723 / 5104 (72.9%)
Hosts with host_response_rate            : 3723 / 5104 (72.9%)
Hosts with host_is_superhost             : 5104 / 5104 (100.0%)
Hosts with host_identity_verified        : 5104 / 5104 (100.0%)
Listings with cancellation_policy   : 5555 / 5555 (100.0%)

  D3 — inclusion criteria
Professional hosts (>1 listing)     : 2594 / 5104 (50.8%)
Listings eligible for D3 analysis   : 2368
Missing availability_30          : 0  (expected: 0)
Missing availability_60          : 0  (expected: 0)
Missing availability_90          : 0  (expected: 0)
Missing availability_365       

This verification confirms that:
- Price outliers are retained as planned
- Missing `cleaning_fee` was properly imputed
- The city populations for D1 are correct
- The inclusion criteria for D2 and D3 match expectations

# 5. Indexing

In [150]:
hosts = db["hosts"]
reviews = db["reviews"]

for col_name, col in [("LISTINGS", collection), ("REVIEWS", reviews), ("HOSTS", hosts)]:
    print(f"\n{'='*50}")
    print(f"  {col_name}")
    print(f"{'='*50}")
    for name, info in col.index_information().items():
        print(f"  {name}: {info['key']}")



  LISTINGS
  _id_: [('_id', 1)]
  price_1: [('price', 1)]
  reviews_1: [('reviews', 1)]
  description_text_name_text: [('_fts', 'text'), ('_ftsx', 1)]
  price_-1: [('price', -1)]

  REVIEWS
  _id_: [('_id', 1)]

  HOSTS
  _id_: [('_id', 1)]


In [151]:
print("\nDeleting indexes on listings...")
collection.drop_index("price_-1")
collection.drop_index("price_1")
collection.drop_index("reviews_1")


print("\nCreating indexes on listings...")

collection.create_index(
    [("price", 1)],
    name="idx_listing_price_full"
)

collection.create_index(
    [("address.market", 1)],
    name="idx_listing_market"
)

collection.create_index(
    [("host_id", 1)],
    name="idx_listing_host_id"
)

collection.create_index(
    [("address.market", 1), ("price", 1)],
    name="idx_listing_market_price"
)

collection.create_index(
    [("property_type", 1)],
    name="idx_listing_property_type"
)

collection.create_index(
    [("address.market", 1), ("property_type", 1)],
    name="idx_listing_market_property"
)

collection.create_index(
    [("review_scores.review_scores_rating", 1)],
    name="idx_listing_rating"
)
print("Listings indexes created.")


print("\nCreating indexes on reviews...")
reviews.create_index(
    [("listing_id", 1)],
    name="idx_reviews_listing_id"
)

reviews.create_index(
    [("date", -1)],
    name="idx_reviews_date_desc"
)

reviews.create_index(
    [("listing_id", 1), ("date", -1)],
    name="idx_reviews_listing_date"
)
print("Reviews indexes created.")


print("\nCreating indexes on hosts...")
hosts.create_index(
    [("host_id", 1)],
)

hosts.create_index(
    [("host_listings_count", 1)],
    name="idx_hosts_listings_count"
)

print("Hosts indexes created.")


Deleting indexes on listings...

Creating indexes on listings...
Listings indexes created.

Creating indexes on reviews...
Reviews indexes created.

Creating indexes on hosts...
Hosts indexes created.


In [152]:
for col_name, col in [("LISTINGS", collection), ("REVIEWS", reviews), ("HOSTS", hosts)]:
    print(f"\n{'='*50}")
    print(f"  {col_name}")
    print(f"{'='*50}")
    for name, info in col.index_information().items():
        print(f"  {name}: {info['key']}")


  LISTINGS
  _id_: [('_id', 1)]
  description_text_name_text: [('_fts', 'text'), ('_ftsx', 1)]
  idx_listing_price_full: [('price', 1)]
  idx_listing_market: [('address.market', 1)]
  idx_listing_host_id: [('host_id', 1)]
  idx_listing_market_price: [('address.market', 1), ('price', 1)]
  idx_listing_property_type: [('property_type', 1)]
  idx_listing_market_property: [('address.market', 1), ('property_type', 1)]
  idx_listing_rating: [('review_scores.review_scores_rating', 1)]

  REVIEWS
  _id_: [('_id', 1)]
  idx_reviews_listing_id: [('listing_id', 1)]
  idx_reviews_date_desc: [('date', -1)]
  idx_reviews_listing_date: [('listing_id', 1), ('date', -1)]

  HOSTS
  _id_: [('_id', 1)]
  host_id_1: [('host_id', 1)]
  idx_hosts_listings_count: [('host_listings_count', 1)]


# A. Price Analytics Expert

## A1. Price Analysis

Analyse prices of properties in three cities: Hong Kong, Montreal, and Barcelona.
Assume that three people will stay for one week. Based on this assumption, calculate the price per person per night, including any additional charges such as cleaning fee etc.

Create box plots for five property types - Apartment, Guesthouse, Hostel, House, Townhouse - for each city, mark the range of prices per person per night, make sure to include min and max prices, mark the median and the range where 50 % of prices lie. 

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning. 2. one graph with 15 box plots where each box plot corresponds to a rating category and contains the distributions for the three cities side by side for comparison. 3. result analysis and interpretation (250-350 words).

## A2. Query Pipeline Analysis

You will calculate the accommodation prices in New York per property type. You are only interested in those properties that have all of the following: Wifi, Hair dryer, 24-hour check-in, Air conditioning, Bed linens. 

Among the selected properties, distinguish between two categories:
    
        A. Wheelchair-friendly properties, defined as those that include at least one of the following amenities: Elevator, Ground floor access, Wheelchair accessible, Wide doorway, Disabled parking spot; 
        B. Family-friendly properties, defined as those that include at least one of the following amenities: Crib, Family/kid friendly, Stair gates, Table corner guards.

Assume that three people will stay for one week. Based on this assumption, the aggregation pipeline should calculate the price per person per night, including any additional charges such as cleaning fee, extra person fee, child fee, or other applicable fees. Calculate the prices for both groups of properties, and both customer categories (wheelchair-, and child-friendly). Transform the output into easy-to-read comparison table.

Create visual with box plots for five most numerous property types for each category (wheelchair- and family-friendly cases side-by-side) on x-axis, and range of prices per person per night on y-axis. Include Median, interquartile range (50% of cases), min/max, outliers.

As this query will be regularly run on this subset, use 'explain' function, to analyse which sequence of pipeline stages, indexes (if any), and any other methods improve your query performance. Output a table with metrics like execution time, docs examined, keys examined, stage, memory, etc to compare raw pipeline query and improved query after you apply different methods to optimize it.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. data interpretation. 2. visual with 10 box plots (5 types * 2 categories). 3. table with comparison metrics of before and after query performance. 4. result analysis and interpretation (max 200 words).


## A3. Analytical Price Suggestion

A number of properties in the collection are missing prices. Analyse those properties, define similar properties where the price exists, and suggest a specific value to the hosts of properties without prices. You can choose to consider property type, location, host details, number of rooms and/or bathrooms, amenities, house rules, etc that indicate the type of clients that those properties attract. 

Another possible avenue to define similar properties could be to analyze the raw reviews using sentiment analysis external to mongodb (e.g. chatgpt can assess the sentiment of review text), and add additional attributes that help you define similar properties. The method and attributes you use to suggest the price is up to you. Keep a clear record of how you did it (a short description before the calculation cell).

As the result, each property that was missing a price should have a new field 'price_suggestion' with a value, all assumptions made and calculation elements considered should be listed, described, and explained clearly. 

Considering only the suggested prices, create box plots for five property types - Apartment, Guesthouse, Hostel, House, Townhouse - for Hong Kong, Montreal, and Barcelona city, mark the range of prices per person per night, make sure to include min and max prices, mark the median and the range where 50 % of prices lie. 

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. reference properties used to estimate suggested price. 2. one graph with 15 box plots where box plots are grouped by property type (not by city). 3. result analysis and interpretation (250-350 words).

# B. Rating Analytics Expert

## B1. Score Analysis

Analyse the following rating categories of properties in three cities: Hong Kong, Montreal, and Barcelona: review_scores_cleanliness, review_scores_accuracy, review_scores_communication, review_scores_location, and review_scores_value.

Create box plots for each rating category across the three cities, making sure to include minimum and maximum values, the median score, and the range where 50% of the scores lie. Each box plot should represent one rating category in one city. The result should be one graph with the fifteen box plots, where each box plot corresponds to a rating category and contains the distributions for the three cities side by side for comparison.

Write a result interpretation, highlighting patterns such as differences in median scores between cities, variability within each rating category, and any notable outliers. Discuss potential reasons for the observed trends and how the rating distributions compare across the cities.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning. 2. one graph with 15 box plots where box plots are grouped by rating category (not by city). 3. result analysis and interpretation (250-350 words).

## B2. Review Quality and Activity Score

Analyse review quality and activity for each property type in New York. Only consider properties with at least 10 reviews. For each property, calculate a Review Quality and Activity Score defined as:

Score = (average overall rating) × 0.5 + (normalized review count) × 0.3 + (recency score) × 0.2
 
This metric combines review quality, popularity, and recency of activity, where: 

    Average overall rating = review_scores_rating 
    Normalized review count = property review count ÷ maximum review count within each property type group e.g. Apartment, or House.
    Recency score = number of days between the latest review in the database and the latest review of the property, scaled between 0 and 1. So that the property with the more recent review in the database will get a score of 1.

Use an aggregation pipeline to compute the Review Quality and Activity Score for each property, and property type group. Transform the output into a readable comparison table showing the score per property type. This query will be repeatedly ran every week to recalibrate the score, so take that into account when you make database and query design decisions.

Create a visual with box plots for ten most numerous property types (x-axis) to show Review Quality and Activity Score (y-axis) of each property type, with median score, min and max values, interquartile range (50% of scores) and outliers.

Use the explain function to analyse the pipeline’s performance. This query will be ran on a regular basis to keep the score relevant. Evaluate which query structures, stages, patterns, and indexes improve execution efficiency. Produce a comparison table with metrics such as execution time, documents examined, keys examined, stage type, and memory usage for before and after the improvements in query performance that you introduced.

The delivered assignment should have three components: 1. assumptions and data decisions (e.g., handling missing or incorrect data). 2. visual with box plots for ten property type. 3. comparison table of query performance before and after optimization. 4. interpretation of both results (max 200 words).

## B3. Long-Term Rental Suggestion
   
Some properties in the collection may experience periods with little or no tenant activity during the year. Using the review dates in the dataset, analyse patterns of tenant presence and identify periods when properties are likely to be unoccupied. Assume that a review is typically written shortly after a stay, therefore the presence of reviews in a given period can be used as an indicator that the property was occupied. Conversely, long gaps between reviews may indicate that the property was empty.

To make the analysis more robust, examine tenant presence separately for each year in the dataset, if more than one year review data exists. Identify periods of potential vacancy within each year and then average the results across years in order to detect recurring vacancy periods more reliably. This analytical query will be updated on the yearly basis.

Analyse these patterns and identify periods when properties appear to be vacant for an extended time. If a property has more than one consecutive month without reviews, and this pattern appears consistently across years, you should recommend that the host consider renting the property long-term during that period. Specifically, create a new field called 'rental_suggestion' that will host an array of month number suitable for long-term rental. 

Create a scatter plot to show which your long-term rental suggestions for Hong Kong, Montreal, and Barcelona. Represent the results in a scatter plot with x axis being months of the year, with y axis being number of properties available during that month, the size of the points should reflect the number of properties available during each month in each city, and finally mark each city with a different color.  

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. assumptions about time gap calculations, any database design choices you made, etc. 2. one scatter plot for Hong Kong, Montreal, and Barcelona rental suggestions. 3. result analysis and interpretation (250-350 words).

# C. Comfort Analytics Expert

## C1. Amenity Premium Analysis

Analyse properties in three cities: Hong Kong, Montreal, and Barcelona to determine which amenities are associated with higher prices per night. Assume two guests stay for one week when calculating the effective price per person per night, including any additional fees such as cleaning fees, extra person charges, or other applicable costs.

Examine all amenities in the dataset to identify those associated with higher price per night, excluding amenities present in all properties since they offer no variation. Each property has a single price, so analyse the statistical association between amenity presence and price. For each amenity, divide properties into two groups—those with the amenity and those without—and compare their price distributions using median price. The difference in median indicates the 'price premium' of the amenity. Are there any amenities that correlate to drop in the price?

Create box plots for all three cities with the three amenities which correlate with the price premium the most in that city. For each amenity case, create two box plots - one for properties with the amenity and one for properties without the amenity (color coded accordingly), with x-axis being amenity in each city, and y-axis being price per person per night. Make sure to include minimum and maximum prices, the median, and the range where 50% of prices lie.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning. 2. one graph with 18 box plots (3 cities x 3 top amenities x 2 cases of with and without). 3. result analysis and interpretation (250-350 words).

## C2. Luxury Amenity of Top-Rated Properties

Identify the amenities most common among top-rated properties and quantify their prevalence. Analyse properties in Sydney and New York to identify the amenities most common among top-rated listings. The analysis will focus on the following group of properties that we will treat as equivalent to each other: Aparthotel, Apartment, Condominium, Loft, and Serviced apartment. 

Only consider properties with at least 30 reviews to ensure reliable statistics, and top 15 % of review_scores_rating (top-rated properties). Among the selected properties, differentiate between two amenity categories: 1. essential amenities group that appear in all or almost all (85 %) properties, and 2. optional luxury amenities group that are more rare. From now on, only consider the optional luxury amenities.

Prevalence of each luxury amenity per city can be calculated as following (you could add more considerations, accompanied by a clear reasoning):

Prevalence = 100 * (Number of properties with amenity) / (Total number of properties) 

Create a double lollipop chart for top 10 amenities in each city (so 20 in total), where the difference in luxury amenity prevalence is visualized between two cities. X-axis holds 20 amenities, y-axis holds prevalence score, and double lollipop stands for each color-coded city.

Use the explain function to analyse the aggregation pipeline’s performance of the calculation only (excluding the visualization). Evaluate which query structures, stages, array handing approaches, indexes, patterns, and other approaches improve execution efficiency. Produce a comparison table with metrics such as execution time, documents examined, keys examined, stage type, and memory usage for before and after the improvements in query performance that you introduced.

The delivered assignment should have three components: 1. assumptions and decisions. 2. a lollipop chart with 10 top luxury amenities and their prevalences per city. 3. comparison table of query performance before and after optimization. 4. interpretation of results (max 200 words).

## C3. Amenity Value Analysis and Suggestion

Some properties in the collection may offer superior guest experiences at a relatively low cost, while others receive lower value ratings. Using the dataset, analyse which amenities are associated with high guest-perceived value. Focus on Hong Kong and Barcelona as your locations.

Use the existing review_scores_value field as a measure of guest satisfaction with value, but combine it with price per person per night with any other considerations that you think are important to define a weighted value metric. When calculating the price, use assumption of two people staying for a week, and include any fees such as cleaning, extra person, into the price calculation. You are responsible for deciding the relative weights of rating versus price, clearly explain and justify your choice before performing the analysis.

Once your weighted value metric is defined:

    1. Examine all amenities in the dataset to determine which ones are associated with higher weighted value scores. Exclude amenities present in more than 85 % of all properties, as they provide no meaningful variation. For each amenity, compare properties with versus without the amenity to measure its contribution to your metric.

    2. Identify properties with below-median weighted value scores in each city. Based on the amenities associated with higher value, suggest specific amenities that these lower-scoring properties could add to improve perceived value. Provide reasoning for each suggestion.

To visualize the results, create box plots showing three most impactful amenities contributing to higher weighted value for Hong Kong and Barcelona. Each box plot should represent one amenity, with one box for properties that have the amenity and one for properties that do not, using color to distinguish between the two. The x-axis should list the amenities, the y-axis should display the weighted value metric, and different panels or labels should represent the two cities. Make sure to include the minimum and maximum values, the median, and the range where 50 % of the values lie.

Finally, provide a concise interpretation and set of host recommendations, impact of weighted score assumptions, highlighting which amenities provide the most guest-perceived value, how patterns differ across cities, and what improvements could be made for low-value properties.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. assumptions and components of weighted value score calculations, any database design choices you made, etc. 2. one visual with 12 box plots (2 cities * 3 amenities * 2 value cases with and without) for Hong Kong and Barcelona. 3. result analysis and interpretation (250-350 words).

# D. Host Performance Expert

## D1. Host Type and Pricing

Analyse properties in three cities: Hong Kong, Montreal, and Barcelona, to determine how the host type influences pricing. Assume two guests stay for one week when calculating the effective price per person per night, including any additional fees such as cleaning fees, extra person charges, or other applicable costs.

Classify hosts into two groups:
        
        1. Professional Hosts: Those who manage more than one property.
        2. Amateur Hosts: Those who manage only one property.

Examine the price distribution within each of the two host groups and calculate the median price per person per night for both. Compare the price distributions to identify any price differences between professional and amateur hosts. Does being a professional host lead to higher or lower prices?

To assess the impact, create one visual with box plots comparing the price per person per night between professional and amateur hosts for each city. For each city, create two box plots: one for professional hosts and one for amateur hosts, using color coding for clarity. The x-axis should represent the host type, and the y-axis should represent the price per person per night. Ensure that you include minimum and maximum prices, median, and the interquartile range (50% of the cases).

The delivered assignment should include the following components: 1. any assumptions and decisions made at the beginning. 2. one graph with 6 box plots (3 cities x 2 host types). 3. result analysis and interpretation (250-350 words).

## D2. Host Features Impact on Satisfaction

The goal is to test the hypothesis that certain host features have a direct impact on customer satisfaction (measured by review score). Specifically, consider the following host features: Superhost status (host_is_superhost), Response time (host_response_time), Host verification status (host_is_verified), Cancellation policy (cancellation_policy), Response rate (host_response_rate).

Customer satisfaction is measured using the review score (review_scores_rating). If the host manages multiple properties, take the average score across listings. Since not all attributes have numerical binary value type, you would need to represent the meaning in strings into relative scores, and choose how to make a split of values into binary. E.g. response time should be translated into whether or not the host is a fast-responder, or not.  

Satisfaction difference score based on each of host features: For each feature (superhost status, response time, etc.) calculate the score. You could calculate using the following example: 

Satisfaction difference = Average Satisfaction for Superhosts - Average Satisfaction for Non-Superhosts

You can add a weight to adjust the score calculation. In which case, make sure to state that in the assumptions. 

Create a double lollipop chart for each host attribute considered in the task. Visualise the difference in the score per attribute. X-axis should hold attributes, y-axis should hold the (weighted) score, and double lollipop stands for each binary value.

Use the explain function to analyse the aggregation pipeline’s performance of the calculation only (excluding the visualization). Evaluate which query structures, stages, array handing approaches, patterns, indexes, and other approaches improve execution efficiency. Produce a comparison table with metrics such as execution time, documents examined, keys examined, stage type, and memory usage for before and after the improvements in query performance that you introduced.

The delivered assignment should have three components: 1. assumptions and decisions. 2. a lollipop chart with 5 host attributes and their scores. 3. comparison table of query performance before and after optimization. 4. interpretation of results (max 200 words).

## D3. Property Popularity and Satisfaction

The goal of this task is to analyze how property unavailability (calculated as the complement of availability) is influenced by guest satisfaction (as indicated by review scores) for professional hosts (those managing more than one property). Assume two guests stay for one week when calculating the effective price per person per night, including any additional fees such as cleaning fees, extra person charges, or other applicable costs.

Unavailability will serve as a measure of property popularity — the more unavailable a property is, the more likely it is booked and thus is in demand. The database contains availability of the property measured in days for four different periods: next 30 days, next 60 days, next 90 days, and next 365 days. You need to suggest a way to calculate popularity based on how many days the property is unavailable in the future. 

The base assumption is that professionally rented properties can only be unavailable because they are booked. Consider adding any further attributes as weights. Keep a clear record of how you did it (a short description before the calculation cell). Based on your calculation, create a new attribute 'popularity_score' in the database. This attribute will be regularly updated in the database, so just your database design and query optimizations accordingly.

Analyze the relationship between satisfaction, and popularity by grouping properties based on guest satisfaction (review rating score, or multiple scores - your choice) in low, medium, and high satisfaction categories. Calculate and compare median popularity score for each satisfaction category. Feel free to include other considerations into this calculation.

As the result, visualize popularity scores per satisfaction category for three most common property types. Create one visual with box plots mark the range of popularity score for each satisfaction category, make sure to include min and max prices, mark the median and the range where 50 % of prices lie. 

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. popularity score calculation. 2. one graph with 9 box plots (3 property types x 3 satisfaction categories) with distribution of popularity score. 3. result analysis and interpretation (250-350 words).

# E. Property Features Expert

## E1. Property Size and Pricing

Analyze the relationship between property size and pricing in three cities: Hong Kong, Montreal, and Barcelona. Specifically, we will examine how the property size (number of rooms and bathrooms / max number of people that can be accommodated) influences its price per person per night.

To calculate price, assume max allowed number of guests stay for one week, and include any additional fees such as cleaning fees, extra person charges, or other applicable costs when calculating the effective price per person per night.

Group the properties by their size given max number of people that can stay there into small, medium, and large. Add any further considerations or attributes. Compare the price per person per night for each of the three property size categories. Create a new attribute 'size_category' and add to the database.

Create one visual with 9 box plots comparing the price per person per night across the three property size groups in each of the three cities. The x-axis of each box plot should represent the property size category, and the y-axis should represent the price per person per night. Ensure that the box plots display the minimum and maximum prices, the median price, the interquartile range (50% of the cases), and any outliers.

The delivered assignment should include the following components: 1. any assumptions and decisions for the size calculation. 2. one graph with 9 box plots (3 cities x 3 size categories). 3. result analysis and interpretation (250-350 words).

## E2. Multi-property Ownership and Bookings

Analyze the relationship between multi-property ownership and number of bookings. Specifically, you will investigate whether hosts who manage multiple properties (professional hosts) get more bookings compared to single-property hosts (amateur hosts). Consider Hong Kong, Montreal, and Barcelona only. Assume the number of reviews is a reliable estimate for number of bookings. 

To begin with, classify hosts into two groups: 1. Professional Hosts (manage more than one property), 2. Amateur Hosts (manage only one property). Add this Boolean attribute to each listing. Compare the booking frequencies (i.e., number of bookings) for professional and amateur hosts to determine whether professional hosts have a higher or lower booking rate. Choose the time period for your comparison across all properties. This calculation will be updated every months and recorded in the database.

Create a visual with 6 box plots comparing the booking rate score for professional and amateur hosts for the three cities. The x-axis should represent the host type (professional vs. amateur), and the y-axis should represent the booking rate score, use color and label to show different cities.

Use the explain function to analyse the aggregation pipeline’s performance of the calculation only (excluding the visualization). Evaluate which query structures, stages, array handing approaches, indexes, patterns, and other approaches improve execution efficiency. Produce a comparison table with metrics such as execution time, documents examined, keys examined, stage type, and memory usage for before and after the improvements in query performance that you introduced.

The delivered assignment should have three components: 1. assumptions and decisions. 2. a visual with 6 box plots. 3. comparison table of query performance before and after optimization. 4. interpretation of results (max 200 words).

## E3. Property Size and Property Comfort

Analyze how property size and property type influence guest satisfaction (as indicated by review scores) for amateur hosts (those managing only one property). Assume two guests stay for one week when calculating the effective price per person per night, including any additional fees such as cleaning fees, extra person charges, or other applicable costs.

Property size will be measured by the number of rooms and/or bathrooms in each listing divided by max number of guests allowed. Sort all properties by their size into three groups: small, medium, and large. Create a new attribute 'size_group' for each listing.

Also, group properties into categories by their similarity in comfort. It is up to you to define which attributes to use, e.g. amenities + room_type, number of bathrooms per person, property types of similar nature - Apartment, Aparthotel, Hostel.

Create a visual with 15 box plots comparing five most common comfort categories for each property size group (3 groups). The x-axis should represent the comfort category per each size group, and the y-axis should represent the satisfaction score.

The delivered assignment should have three components: 1. any assumptions and decisions made at the beginning, e.g. popularity score calculation. 2. one graph with 15 box plots (3 property size groups x 5 comfort categories) with distribution of satisfaction score. 3. result analysis and interpretation (250-350 words).